# Lesson 01 - Įvadas į DI agentus

Sveiki atvykę į pirmąjį pamoką kursuose **DI agentai pradedantiesiems**!

**DI agentas** yra programa, kuri naudoja didelį kalbos modelį (LLM) kaip savo samprotavimo variklį ir gali imtis *veiksmų* realiame pasaulyje — kviesti API, užklausinėti duomenų bazes ar vykdyti kodą — norėdama pasiekti tikslą naudotojo vardu.

Šiame užrašų knygelyje sukursite savo pirmąjį agentą: **Kelionių agentą**, kuris rekomenduoja atostogų kryptis. Kelyje sužinosite, kaip:

1. Prisijungti prie Azure AI Foundry Agent Service naudojant **Microsoft Agent Framework**.
2. Paskirti agentui **įrankį** — paprastą Python funkciją, kurią jis gali kviesti.
3. Paleisti agentą ir peržiūrėti jo atsakymą.
4. Srautiniu būdu perduoti agento atsakymą po vieną tokeną.


## Nustatymas

Prieš paleisdami šį užrašų knygelę, įsitikinkite, kad turite:

1. **Azure AI Foundry projektą** su diegta pokalbių modeliu (pvz., `gpt-4o-mini`).
2. **Prisijungę naudodami Azure CLI** — terminale paleiskite `az login`.
3. **Nustatytus reikalingus aplinkos kintamuosius:**
   - `AZURE_AI_PROJECT_ENDPOINT` — jūsų Azure AI Foundry projekto galinis taškas.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jūsų diegto modelio pavadinimas.

Žemiau esanti ląstelė įdiegia jums reikiamus Python paketus.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kurkite savo pirmąjį agentą

Agentui reikia dviejų dalykų:

- **Instrukcijų**, kurios nurodo *kas jis yra* ir *kaip elgtis* (sistemos užklausa).
- **Įrankių** — Python funkcijų, papuoštų `@tool`, kurias agentas gali kviesti norėdamas gauti informaciją arba atlikti veiksmus.

Žemiau apibrėžiame paprastą įrankį, kuris grąžina populiarių atostogų vietų sąrašą. Agentas naudos šį įrankį, kai vartotojas paprašys kelionės rekomendacijų.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Srautiniai atsakymai

Dėl interaktyvesnės patirties galite **srautu** gauti agento atsakymą. Vietoj to, kad lauktumėte viso atsakymo, agentas pateikia teksto dalis, kai jos sukuriamos. Tai ypač naudinga pokalbių sąsajose, kur norite realiu laiku rodyti rezultatus.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Santrauka

Šiame pamokoje sužinojote, kaip:

- **Sukurti tiekėją**, kuris jungiasi prie Azure AI Foundry Agent Service per `AzureAIProjectAgentProvider`.
- **Apibrėžti įrankį** naudojant dekoratorių `@tool`, kad agentas galėtų kviesti jūsų Python funkcijas.
- **Paleisti agentą** su vartotojo žinute ir atspausdinti jo atsakymą.
- **Srautinį atsakymų perdavimą**, kad būtų išvestis realiu laiku.

Kitoje pamokoje gilinsimės į agentinius karkasus ir sužinosime, kaip suteikti agentams galingesnius įrankius bei daugiapakopį mąstymo gebėjimą.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės atsisakymas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatiniais vertimais gali būti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turi būti laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogaus vertimą. Mes neatsakome už bet kokius nesusipratimus ar neteisingus interpretavimus, kilusius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
